# Модули в Python

Когда программа разрастается до сотен строк, держать весь код в одном файле становится неудобно. Модули позволяют разнести код по нескольким файлам и подключать нужные части туда, где они требуются.

## Что такое модуль?

<b>Модуль в Python</b> - это просто файл с расширением .py, содержащий код Python (функции, классыБ переменные), который можно импортировать и использовать в других программах.

## Создание собственного модуля и его импорт

Создать модуль в Python очень просто: достаточно написать код в файле с расширением <b>.py</b>. Заведём <b>mymath.py</b> и положим рядом несколько <b>main</b>-скриптов, которые показывают разные способы его подключить.

<b>Структура проекта:</b>
- main_aliased.py
- main_basic.py
- main_specific.py
- main_star.py
- mymath.py

В дереве слева наш модуль mymath.py и четыре main_*.py, каждый показывает свой способ импорта. Переключайтесь между вкладками. Ниже короткий разбор, когда какой удобнее.

### Импорт всего модуля
Самый прямой способ: <b>import mymath</b> (см. <b>min_basic.py</b>). Все имена остаются "внутри" модуля и доступны через префикс: <b>mymath.add, mymath.PI</b>. Удобно, когда из одного модуля нужно много функций и важно видеть, откуда они пришли.

### Импорт конкретных элементов
<b>from math import add, multiply</b> (см. <b>main_specific.py</b>). Импортируем только нужное и обращаемся без префикса. Хорошо для пары часто вызываемых функций; плохо, когда через сотню строк забываешь, откуда взялась <b>add</b>.

### Импорт с переименованием
<b>import mymath as mm</b> (см. <b>main_aliased.py</b>). Нужно, если имя модуля длинное (<b>numpy as np, pandas as pd</b> те самые случаи) или конфликтует с локальной переменной.

### Импорт всех элементов
<b>from mymath import *</b> (см. <b>main_star.py</b>). Импортирует всё подряд в текущее пространство имён. В обычном коде так делать обычно не стоит: теряется источние имён, легко получить конфликт. Нормально только в REPL и итогда в тестах.

## Где Python ищет модули
Когда вы пишете <b>import mymath</b>, Python ищет файл <b>mymath.py</b> по списку директорий из <b>sys.path</b>. По умолчанию туда входят:
- Директория, из которой запущен скрипт (или текущая директория в REPL)
- Встроенные модули (<b>math, os, ...</b>) - они часть Python
- site-packages - папка, куда <b>pip install</b> устанавливает сторонние пакеты

Поиск останавливается на первой найденной директории. Это значит, что если рядом со скриптом лежит файл с тем же именем, что и стандартный модуль, Python подхватит ваш, а не системный. Классическая ловушка: создать <b>random.py</b> в проекте и потом долго удивляться, почему <b>random.randint</b> не работает.

Содержимое <b>sys.path</b> можно посмотреть так:

In [2]:
# import sys
# print(sys.path)

## Специальные переменные модуля
В Python модули имеют несколько специальных переменных.

### Переменная --name--: модуль как программа vs как зависимость

Внутри Python у каждого модуля есть переменная --name--. Когда модуль импортируют, в ней лежит его имя (<b>"mymath"</b>). А когда модуль запускают напрямую (<b>python mymath.py</b>), Python кладёт в неё специальное значение <b>"--main--"</b>. Это позволяет внутри модуля написать блок "делай это только при прямом запуске":

<b>Структура проекта:</b>
- main.py
- mymath.py

In [ ]:
# mymath.py
"""Модуль с математическими функциями."""

PI = 3.14159

def add(a, b):
    return a + b

def divide(a, b):
    if b == 0:
        raise ValueError('Деление на ноль невозможно')
    return a / b

# Этот блок выполняется только при прямом запуске:
#   python mymath.py
# При импорте (import mymath) в другом файле он не сработает.
if __name__ == '__main__':
    print(f'PI = {PI}')
    print(f'add(2, 3) = {add(2, 3)}')

In [ ]:
# main.py
# При импорте __name__ внутри mymath равно "mymath",
# поэтому if-блок в самом mymath.py молчит.
import mymath
print(mymath.add(5, 3))

Запустите <b>python mymath.py</b>, и увидите вывод if-блока. Запустите <b>python main.py</b>, и этот блок промолчит: в выводе будет только <b>8</b> от вызова в <b>main.py</b>.

### Переменная --all--: что попадает в from module import *

По умолчанию <b>from mymath import *</b> импортирует все имена, которые не начинаются с подчёркивания. Если хочется явно зафиксировать публичный API модуля, добавляют переменную <b>--all--</b> со списком имён.

<b>Структура проекта:</b>
- main_star.py
- mymath.py

In [ ]:
# mymath.py
"""Модуль с математическими функциями."""
 
# Только эти имена попадут в from mymath import *
__all__ = ["PI", "add"]
 
PI = 3.14159
_INTERNAL = "не должен попасть наружу"
 
def add(a, b):
    return a + b
 
def subtract(a, b):
    return a - b
 
def _round_helper(value):
    """Внутренний помощник, не для публичного использования."""
    return round(value, 2)

In [ ]:
# main_star.py
from mymath import *
 
# Доступны: PI и add
print(PI)         # 3.14159
print(add(1, 2))  # 3
 
# А вот subtract в текущем пространстве нет:
# print(subtract(5, 3))  # NameError: name 'subtract' is not defined

Через <b>from mymath import *</b> пришли только <b>PI</b> и <b>add</b>, именно те, что перечислены в <b>--all--</b>. Остальные имена (<b>subtract, _INTERNAL, _round_helper) остались в модуле и доступны только при явном импорте: <b>from mymath import subtract</b>.

### Пакеты

Когда модуль <b>mymath.py</b> разрастается (например, появляются разные группы операций), его можно разделить на несколько файлов и собрать в <b>пакет</b>. Пакет в Python это директория с файлом <b>--init--.py</b>. Когда вы пишете <b>import mathlib</b>, Python видит этот <b>--init--.py</b> и понимает, что директория это импортируемый пакет.

Превратим наш модуль в пакет: разнесём функции по тематикам, а в <b>--init--.py</b> соберём публичный интерфейс.

<b>Структура проекта:</b>
- mathlib
    - --init--.py
    - advanced.py
    - basic.py
- main.py

In [ ]:
# main.py
# Можно обращаться к подмодулю напрямую
import mathlib.basic

print(mathlib.basic.add(2, 3))

# А благодаря re-export'у в __init__.py короче:
from mathlib import add, divide

print(add(10, 20))
print(divide(10, 4))

In [ ]:
# __init__.py
"""mathlib: пакет с математическими функциями."""
 
__version__ = "0.1"
 
# Re-export, чтобы пользователи могли писать from mathlib import add
from mathlib.basic import PI, add, subtract
from mathlib.advanced import multiply, divide

In [ ]:
# advanced.py
"""Операции, которые могут поднять исключение."""
 
def multiply(a, b):
    return a * b
 
def divide(a, b):
    if b == 0:
        raise ValueError("Деление на ноль невозможно")
    return a / b

In [ ]:
# basic.py
"""Базовые операции."""
 
PI = 3.14159
 
def add(a, b):
    return a + b
 
def subtract(a, b):
    return a - b

Что изменилось:
- Файлов теперь четыре, но снаружи это всё ещё "один математический модуль": пользователь пишет <b>from mathlib import add</b>, как и раньше с <b>mymath</b>.
- <b>mathlib/--init--.py</b> контролирует, что считается публичным API: имена, перечисленные через <b>from .basic import...</b> и <b>from .advanced import ...</b>, достпны прямо как <b>mathlib.add</b>.
- Внутри пакета модули могут ссылаться друг на друга через относительные импорты: <b>from . import basic</b> (тот же каталог), <b>from .. impprt other</b> (родительский каталог).

Если пакет растет ещё, его можно разбить на подпакеты: например, <b>mathlib/stats/</b> со своим <b>--init--.py</b> и модулями статистических функций. Принцип тот же, просто на уровень глубже.

### Как организовать модуль

Несколько практик, которые делают модуль удобным и для вас, и для других читателей.
<b>Одна ответственность</b>. Каждый модуль отвечает за одну конкретную задачу: <b>data_processing.py, auth.py, formatters.py</b>. Если в файле уже две несвязанные темы, пора разделять.

<b>Понятные имена</b>. Описательные, но короткие, в стиле snake_case: <b>user_interface.py</b>, не <b>ui.py</b> и не <b>UserInterface.py</b>.
Порядок содержимого внутри файла:
- <b>Docstring</b>: описание модуля в тройных кавычках в самом начале.
- <b>Импорты</b>: сначала стандартная библиотека, потом сторонние пакеты, потос собственные модули.
- <b>Константы</b>: глобальные значения, которые не меняются.
- <b>Классы и функции</b>: основное содержимое.
- <b>Блок if --name-- == "--main--"</b>: код, который выполняется только при прямом запуске файла.

<b>Явные импорты.</b> Предпочитайте <b>from module import specific_string</b> вместо <b>from module import *</b>: видно, что именно используется.

<b>Приватные имена с _</b>. Функции и переменные "для внутреннего пользования" начинайте с подчёркивания (<b>_helper</b>, <b>_INTERNAL_CONST</b>). Это сигнал для других: "не полагайтесь на это снаружи".